In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import pickle
from tqdm.auto import tqdm
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, f1_score,
)
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model

print(f'TF version: {tf.__version__}')

OBSERVATIONS_PKL   = '/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl'
EXOPLANNET_H5_PATH = '/kaggle/input/datasets/maanav0114/model-and-baselines-evaluation-data/exoplANNET_trained.h5'  # UPDATE to your dataset path
RESULTS_PKL        = '/kaggle/working/exoplannet_eval_results.pkl'

N_INPUT     = 990      # ExoplANNET's expected input length
N_PEAKS_MAX = 5        # max Virtual Astronomer iterations
THRESHOLDS  = [0.77]               # ExoplANNET's published threshold
F_MAX_DAYS  = 7000.0
MIN_FREQ    = 1.0 / F_MAX_DAYS
MAX_FREQ    = 1.0 / 1.0
FREQ_GRID   = np.linspace(MIN_FREQ, MAX_FREQ, N_INPUT)   # fixed 990-point grid

print('Audit protocol:')
print(f'  periodogram grid: {N_INPUT} bins over [{MIN_FREQ:.6f}, {MAX_FREQ:.4f}] cyc/day')
print(f'  max VA iterations: {N_PEAKS_MAX}')
print(f'  thresholds: {THRESHOLDS}')


assert os.path.exists(OBSERVATIONS_PKL), f'observations.pkl not found at {OBSERVATIONS_PKL} — run data_prep.ipynb first'
observations = pd.read_pickle(OBSERVATIONS_PKL)
print(f'\nobservations.pkl: {observations.shape}, columns={list(observations.columns)}')

star_labels = observations.groupby('star_name')['has_exoplanets'].first()
print(f'  stars: {len(star_labels)} | pos: {(star_labels==1).sum()} | neg: {(star_labels==0).sum()}')

print(f'\nExoplANNET weights: {EXOPLANNET_H5_PATH} ({os.path.getsize(EXOPLANNET_H5_PATH):,} bytes)')


In [ ]:
model = load_model(EXOPLANNET_H5_PATH, compile=False)
print('ExoplANNET loaded.')
print()
model.summary()

print('\nExpected inputs:')
for i, inp in enumerate(model.inputs):
    print(f'  input_{i}: shape={inp.shape}, dtype={inp.dtype}')

print('\nExpected outputs:')
outputs = model.output if isinstance(model.output, list) else [model.output]
for i, out in enumerate(outputs):
    print(f'  output_{i}: shape={out.shape}, dtype={out.dtype}')

print('\nModel.layers (last 8):')
for L in model.layers[-8:]:
    print(f'  {L.name} ({L.__class__.__name__})')

In [ ]:

def compute_periodogram(bjd, rv, freq_grid):
    """Compute GLS periodogram with standard normalization."""
    if len(bjd) <= 1 or np.std(rv) < 1e-12:
        return np.zeros(len(freq_grid), dtype=np.float32)
    try:
        ls = LombScargle(bjd, rv, normalization='standard')
        power = ls.power(freq_grid)
    except Exception:
        return np.zeros(len(freq_grid), dtype=np.float32)
    p = np.asarray(power, dtype=np.float32)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p = np.clip(p, 0.0, None)
    return p

def fit_and_remove_sinusoid(bjd, rv, freq):
    """Fit A·sin(2π f t) + B·cos(2π f t) + C and return residuals."""
    omega = 2 * np.pi * freq
    A_sin = np.sin(omega * bjd)
    A_cos = np.cos(omega * bjd)
    A = np.column_stack([A_sin, A_cos, np.ones_like(bjd)])
    coeffs, _, _, _ = np.linalg.lstsq(A, rv, rcond=None)
    rv_residuals = rv - (coeffs[0] * A_sin + coeffs[1] * A_cos)
    return rv_residuals.astype(np.float64)

def predict_exoplannet_single(model, periodogram, idx_peak, power_peak):
    """Run ExoplANNET on a single peak."""
    pg_input = periodogram.astype(np.float32).reshape(1, N_INPUT, 1)

    aux_input = np.array([[float(idx_peak), float(power_peak)]], dtype=np.float32).reshape(1, 2, 1)

    preds = model.predict([pg_input, aux_input], verbose=0)
    if isinstance(preds, list):
        candidates = [p for p in preds if np.asarray(p).shape[-1] == 1]
        preds = candidates[-1] if candidates else preds[-1]
    return float(np.asarray(preds).reshape(-1)[0])

def virtual_astronomer_exoplannet(model, bjd, rv, freq_grid, threshold=0.77,
                                  max_iterations=N_PEAKS_MAX):
    """Paper's Virtual Astronomer (Algorithm 1) with ExoplANNET scoring."""
    peak_predictions = []
    rv_current = rv.copy().astype(np.float64)

    for iteration in range(max_iterations):
        pg = compute_periodogram(bjd, rv_current, freq_grid)

        max_idx = int(np.argmax(pg))
        max_freq = float(freq_grid[max_idx])
        max_power = float(pg[max_idx])

        if max_freq < MIN_FREQ:
            break

        prob = predict_exoplannet_single(model, pg, max_idx, max_power)

        peak_predictions.append({
            'freq': max_freq,
            'period': 1.0 / max_freq,
            'power': max_power,
            'probability': prob,
        })

        if prob >= threshold:
            rv_current = fit_and_remove_sinusoid(bjd, rv_current, max_freq)
        else:
            break

    star_score = max([p['probability'] for p in peak_predictions]) if peak_predictions else 0.0
    return peak_predictions, star_score

In [ ]:
star_labels_map = star_labels.to_dict()

unique_stars = sorted(star_labels_map.keys())
print(f'Stars in observations.pkl: {len(unique_stars)}')

all_peak_preds  = []         # per-peak probability
all_peak_labels = []         # per-peak: 1 if star has planet, else 0
all_peak_star   = []         # parent star name
all_peak_period = []         # peak period (days)
all_peak_freq   = []         # peak frequency (1/days)
all_peak_power  = []         # peak power (raw GLS standard)

star_pred_max   = {}          # star_name -> max peak prob (primary star-level score)
star_pred_any   = {}          # star_name -> {threshold: 1/0}
star_label_map  = {}          # star_name -> 0/1

n_skipped_const = 0           # stars with constant RV (no periodogram)
n_skipped_few   = 0           # stars with <=1 observation
n_errors        = 0

print('\nRunning ExoplANNET Virtual Astronomer...')
for star_name in tqdm(unique_stars, desc='ExoplANNET VA'):
    try:
        star_df = observations[observations['star_name'] == star_name]
        bjd = star_df['bjd'].values.astype(np.float64)
        rv  = star_df['rv_centered'].values.astype(np.float64)
    except Exception:
        n_errors += 1
        continue

    y_star = int(star_labels_map[star_name])
    star_label_map[star_name] = y_star

    if len(bjd) <= 1:
        n_skipped_few += 1
        continue
    if np.std(rv) < 1e-12:
        n_skipped_const += 1
        star_pred_max[star_name] = 0.0
        star_pred_any[star_name] = {thr: 0 for thr in THRESHOLDS}
        continue

    try:
        peak_predictions, star_score = virtual_astronomer_exoplannet(
            model, bjd, rv, FREQ_GRID, threshold=0.77  # published threshold
        )
    except Exception:
        n_errors += 1
        continue

    for p in peak_predictions:
        all_peak_preds.append(p['probability'])
        all_peak_labels.append(y_star)
        all_peak_star.append(star_name)
        all_peak_freq.append(p['freq'])
        all_peak_period.append(p['period'])
        all_peak_power.append(p['power'])

    star_pred_max[star_name] = star_score
    star_pred_any[star_name] = {thr: int(np.any([p['probability'] > thr for p in peak_predictions]))
                               for thr in THRESHOLDS}

n_total    = len(unique_stars)
n_evaluated = n_total - n_skipped_const - n_skipped_few - n_errors
n_pos_total = sum(1 for s in unique_stars if star_label_map.get(s, 0) == 1)
n_neg_total = sum(1 for s in unique_stars if star_label_map.get(s, 0) == 0)
n_pos_eval  = sum(1 for s in unique_stars if star_label_map.get(s, 0) == 1 and s in star_pred_max)
n_neg_eval  = sum(1 for s in unique_stars if star_label_map.get(s, 0) == 0 and s in star_pred_max)
print('\nCoverage:')
print(f'  total stars: {n_total}')
print(f'  evaluated: {n_evaluated}')
print(f'  skipped (constant RV): {n_skipped_const}')
print(f'  skipped (<=1 obs): {n_skipped_few}')
print(f'  skipped (errors): {n_errors}')
print(f'  Pos coverage                           : '
      f'{100*n_pos_eval/max(1, n_pos_total):.1f}% ({n_pos_eval}/{n_pos_total})')
print(f'  Neg coverage                           : '
      f'{100*n_neg_eval/max(1, n_neg_total):.1f}% ({n_neg_eval}/{n_neg_total})')

n_no_peaks     = n_skipped_const + n_skipped_few + n_errors  # total skipped
n_no_peaks_pos = sum(1 for s in unique_stars
                     if star_label_map.get(s, 0) == 1 and s not in star_pred_max)
n_no_peaks_neg = sum(1 for s in unique_stars
                     if star_label_map.get(s, 0) == 0 and s not in star_pred_max)
print(f"\nClass-split skip breakdown: pos_skipped={n_no_peaks_pos}, neg_skipped={n_no_peaks_neg}")

In [ ]:
star_eval = sorted(star_pred_max.keys())
y_star  = np.array([star_label_map[s] for s in star_eval])
p_star  = np.array([star_pred_max[s]  for s in star_eval])
n_pos   = int((y_star == 1).sum())
n_neg   = int((y_star == 0).sum())

print(f'\nStars evaluated: {len(y_star)} (pos={n_pos}, neg={n_neg})')

print('\nstar-level metrics:')
if n_pos > 0 and n_neg > 0:
    pr_auc  = average_precision_score(y_star, p_star)
    roc_auc = roc_auc_score(y_star, p_star)
    print(f'  pr_auc: {pr_auc:.4f}')
    print(f'  roc_auc: {roc_auc:.4f}')
else:
    pr_auc = roc_auc = None
    print('  one class missing — pr_auc/roc_auc undefined')

print('\nstar-level binary:')
results_by_threshold = {}
for thr in THRESHOLDS:
    binary_pred = np.array([star_pred_any[s][thr] for s in star_eval])
    if binary_pred.sum() == 0 or binary_pred.sum() == len(binary_pred):
        all_pos = binary_pred.sum() == len(binary_pred)
        outcome = 'positive' if all_pos else 'negative'
        print(f'  thr={thr:.2f}: degenerate (all {outcome})')
        continue
    cm = confusion_matrix(y_star, binary_pred)
    tn, fp = cm[0]
    fn, tp = cm[1]
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    print(f'  thr={thr:.2f}: P={prec:.4f} R={rec:.4f} F1={f1:.4f}   '
          f'(TN={tn}, FP={fp}, FN={fn}, TP={tp})')
    results_by_threshold[thr] = {
        'precision': float(prec), 'recall': float(rec), 'f1': float(f1),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

print('\npeak-level:')
peak_preds  = np.array(all_peak_preds)
peak_labels = np.array(all_peak_labels)  # star label, not per-peak truth
peak_results = {}
for thr in THRESHOLDS:
    bin_pred = (peak_preds > thr).astype(int)
    if bin_pred.sum() == 0 or bin_pred.sum() == len(bin_pred):
        continue
    cm = confusion_matrix(peak_labels, bin_pred)
    tnn, fpp = cm[0]
    fnn, tpp = cm[1]
    pp = tpp / (tpp + fpp) if (tpp + fpp) > 0 else 0.0
    rr = tpp / (tpp + fnn) if (tpp + fnn) > 0 else 0.0
    ff = 2 * pp * rr / (pp + rr) if (pp + rr) > 0 else 0.0
    print(f'  thr={thr:.2f}: P={pp:.4f} R={rr:.4f} F1={ff:.4f}   '
          f'(TN={tnn}, FP={fpp}, FN={fnn}, TP={tpp})')
    peak_results[thr] = {
        'precision': float(pp), 'recall': float(rr), 'f1': float(ff),
        'tn': int(tnn), 'fp': int(fpp), 'fn': int(fnn), 'tp': int(tpp),
    }

print(f'\nTotal peaks evaluated: {len(peak_preds)}')

vp, vr, vt = precision_recall_curve(y_star, p_star)
vf1 = 2 * vp * vr / (vp + vr + 1e-8)
bi = int(np.argmax(vf1))
opt_thr = float(vt[bi]) if bi < len(vt) else 0.77
opt_preds = (p_star >= opt_thr).astype(int)
opt_cm = confusion_matrix(y_star, opt_preds)
otn, ofp, ofn, otp = opt_cm.ravel()
opt_p = otp/(otp+ofp) if (otp+ofp)>0 else 0.0
opt_r = otp/(otp+ofn) if (otp+ofn)>0 else 0.0
opt_f1 = 2*opt_p*opt_r/(opt_p+opt_r) if (opt_p+opt_r)>0 else 0.0
print(f"\nstar-level at F1-optimal threshold ({opt_thr:.4f}):")

print(f"  P={opt_p:.4f} R={opt_r:.4f} F1={opt_f1:.4f}")
print(f"  Confusion: TN={otn} FP={ofp} FN={ofn} TP={otp}")

def bootstrap_roc_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for ROC-AUC via the percentile method."""
    from sklearn.metrics import roc_auc_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap ROC-AUC")

    point = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aucs[i] = point  # fall back to point estimate if degenerate
            continue
        aucs[i] = roc_auc_score(yt, ys)
    lo = float(np.percentile(aucs, 2.5))
    hi = float(np.percentile(aucs, 97.5))
    return point, lo, hi

def bootstrap_pr_auc(y_true, y_score, n_resamples=200, seed=42):
    """Bootstrap 95% CI for PR-AUC (average precision) via the percentile method."""
    from sklearn.metrics import average_precision_score

    y_true = np.asarray(y_true).ravel()
    y_score = np.asarray(y_score).ravel()
    if len(y_true) != len(y_score):
        raise ValueError(f"length mismatch: y_true={len(y_true)} y_score={len(y_score)}")
    if len(y_true) < 2:
        raise ValueError("need at least 2 samples to bootstrap PR-AUC")

    point = float(average_precision_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aps = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        sample_idx = rng.integers(0, n, size=n)
        yt = y_true[sample_idx]
        ys = y_score[sample_idx]
        attempts = 0
        while len(np.unique(yt)) < 2 and attempts < 10:
            sample_idx = rng.integers(0, n, size=n)
            yt = y_true[sample_idx]
            ys = y_score[sample_idx]
            attempts += 1
        if len(np.unique(yt)) < 2:
            aps[i] = point  # fall back to point estimate if degenerate
            continue
        aps[i] = average_precision_score(yt, ys)
    lo = float(np.percentile(aps, 2.5))
    hi = float(np.percentile(aps, 97.5))
    return point, lo, hi
pr_point, pr_lo, pr_hi = bootstrap_pr_auc(y_star, p_star)
roc_point, roc_lo, roc_hi = bootstrap_roc_auc(y_star, p_star)
print("\nbootstrap 95% CI (200 resamples):")
print(f"  pr_auc: {pr_point:.4f} [{pr_lo:.4f}, {pr_hi:.4f}]")
print(f"  roc_auc: {roc_point:.4f} [{roc_lo:.4f}, {roc_hi:.4f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(p_star[y_star == 0], bins=50, alpha=0.5,
             label=f'No planet (n={n_neg})', color='steelblue')
axes[0].hist(p_star[y_star == 1], bins=50, alpha=0.5,
             label=f'RV host  (n={n_pos})', color='coral')
axes[0].set_xlabel('max peak probability per star')
axes[0].set_ylabel('count')
axes[0].set_yscale('log')
axes[0].set_title('ExoplANNET — star-level score distribution')
axes[0].legend()

if pr_auc is not None:
    prec, rec, _ = precision_recall_curve(y_star, p_star)
    axes[1].plot(rec, prec, label=f'PR curve (AUC={pr_auc:.4f})', color='coral')
    axes[1].axhline(y_star.mean(), ls='--', color='gray',
                    label=f'class rate ({y_star.mean():.3f})')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('ExoplANNET — star-level PR curve')
    axes[1].legend(); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1.05])
plt.tight_layout()
plt.show()


In [ ]:
results = {
    'coverage': {
        'n_total_eval_stars': len(star_eval),
        'n_evaluated_with_peaks': n_evaluated,
        'n_skipped_no_peaks': n_no_peaks,
        'n_skipped_errors': n_errors,
        'n_no_peaks_positives': n_no_peaks_pos,
        'n_no_peaks_negatives': n_no_peaks_neg,
        'n_pos_total': n_pos_total,
        'n_neg_total': n_neg_total,
        'coverage_positives': (n_pos_total - n_no_peaks_pos) / max(1, n_pos_total),
        'coverage_negatives': (n_neg_total - n_no_peaks_neg) / max(1, n_neg_total),
    },
    'metrics_star': {
        'pr_auc': float(pr_auc) if pr_auc is not None else None,
        'roc_auc': float(roc_auc) if roc_auc is not None else None,
        'by_threshold': results_by_threshold,
    },
    'peak_level': {
        'n_peaks': len(peak_preds),
        'by_threshold': peak_results,
    },
    'star_names_evaluated': star_eval,
    'y_star': y_star.tolist(),
    'p_star_max': p_star.tolist(),
    'thresholds': THRESHOLDS,
    'n_peaks_per_star': N_PEAKS_MAX,
    'peak_level_raw': {
        'star_names': all_peak_star,
        'labels': all_peak_labels,
        'predictions': all_peak_preds,
        'periods': all_peak_period,
        'frequencies': all_peak_freq,
        'powers_norm': all_peak_power,
    },
}

with open(RESULTS_PKL, 'wb') as f:
    pickle.dump(results, f)
print(f'Results saved to {RESULTS_PKL}')
print(f'  size: {os.path.getsize(RESULTS_PKL):,} bytes')

with open(RESULTS_PKL, 'rb') as f:
    check = pickle.load(f)
print(f'  Reload OK, keys: {list(check.keys())}')